In [8]:
import cv2
import numpy as np
from IPython.display import display, clear_output
from PIL import Image
import ipywidgets as widgets
import io

# 1. Carregando a Imagem (Com o seu sistema de fallback)
caminho_da_imagem = 'C:\\Users\\thoma\\OneDrive\\Projetos\\Processamento_de_Imagem_3d\\beneficios-da-maca-scaled.jpg' 
img = cv2.imread(caminho_da_imagem)

if img is None:
    print(f"Aviso: Imagem não encontrada. Gerando imagem padrão...")
    img = np.zeros((400, 600, 3), dtype=np.uint8)
    for i in range(400):
        for j in range(600):
            img[i, j] = [int(255 * i / 400), int(255 * j / 600), 128]
    cv2.circle(img, (150, 100), 50, (0, 255, 255), -1)
    cv2.circle(img, (450, 300), 60, (255, 0, 0), -1)
    cv2.rectangle(img, (50, 250), (200, 350), (0, 255, 0), -1)

altura, largura = img.shape[:2]

# Redimensionamento
max_width, max_height = 600, 500
if largura > max_width or altura > max_height:
    escala = min(max_width / largura, max_height / altura)
    img = cv2.resize(img, (int(largura * escala), int(altura * escala)))
    altura, largura = img.shape[:2]

# 2. Gerando o Mapa de Profundidade
y, x = np.mgrid[0:altura, 0:largura]
centro_x, centro_y = largura / 2, altura / 2
mapa_profundidade = np.exp(-((x - centro_x)**2 + (y - centro_y)**2) / (0.1 * (largura**2 + altura**2)))

forca_maxima = 30 
profundidade_calculada = mapa_profundidade * forca_maxima

print("🎮 Simulador 3D - Controle com Joystick Virtual")
print("=" * 60)
print(f"Dimensões da imagem: {largura}x{altura}\n")

# 3. Função que será chamada pelos sliders com @interact
def exibir_simulacao(dx=widgets.FloatSlider(min=-1.0, max=1.0, step=0.05, value=0.0, description='X (←→):'),
                     dy=widgets.FloatSlider(min=-1.0, max=1.0, step=0.05, value=0.0, description='Y (↑↓):')):
    
    # Calcula os mapas de deslocamento
    mapa_x = np.float32(x - profundidade_calculada * dx)
    mapa_y = np.float32(y - profundidade_calculada * dy)
    
    # Aplica o efeito DIBR
    imagem_deslocada = cv2.remap(img, mapa_x, mapa_y, 
                                 interpolation=cv2.INTER_LINEAR, 
                                 borderMode=cv2.BORDER_REPLICATE)
    
    # Converte para RGB
    img_rgb = cv2.cvtColor(imagem_deslocada, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)
    
    # Exibe a imagem
    display(img_pil)
    
    # Mostra informações do joystick
    print(f"Joystick → X: {dx:+.2f} | Y: {dy:+.2f}")

# Cria a interface interativa com @interact
widgets.interact(exibir_simulacao)

🎮 Simulador 3D - Controle com Joystick Virtual
Dimensões da imagem: 600x383



interactive(children=(FloatSlider(value=0.0, description='X (←→):', max=1.0, min=-1.0, step=0.05), FloatSlider…

<function __main__.exibir_simulacao(dx=FloatSlider(value=0.0, description='X (←→):', max=1.0, min=-1.0, step=0.05), dy=FloatSlider(value=0.0, description='Y (↑↓):', max=1.0, min=-1.0, step=0.05))>